In [1]:
import sys
import os
sys.path.append(os.path.abspath('..')) # Go up one level to the project root and add it to the path
from src.etl.ingest import load_nvdrs
from src.etl.transform import filter_nvdrs_suicides, aggregate_nvdrs_daily

In [2]:
#usecols = ['IncidentID', 'IncidentYear', 'SiteID', 'IncidentNumber', 'IncidentCategory_c', 'HomicideSuicide_c', 'PersonID', 'VictimNumber', 'PersonType', 'NumberWeapons_c', 'NumberSuspects_c', 'NumberSubstances_c', 'NumberSubstancesCausedDeath_c', 'Sex', 'AgeYears_c', 'Country', 'ResidenceState', 'ResidenceFIPS', 'ResidenceCityState', 'ResidenceZip', 'Homeless', 'AbstractorDeathmanner_c', 'InjuryState', 'InjuryFIPS', 'InjuryCityState', 'InjuryZip', 'InjuryDate', 'InjuryDate_myr', 'InjuryDate_year', 'InjuryTime', 'InjuryLocationType', 'RecentRelease', 'AlcoholUseSuspected', 'ExternalCause1ICD9', 'ExternalCause2ICD9', 'UnderlyingCauseCode', 'DeathCause1', 'DeathCause2', 'DeathCause3', 'OtherCondition', 'HowInjuryOccurred', 'DeathDate', 'DeathDate_myr', 'DeathDate_year', 'DeathState', 'DeathFIPS', 'MultipleConditionCode01ICD10', 'MultipleConditionCode02ICD10', 'MultipleConditionCode03ICD10', 'MultipleConditionCode04ICD10', 'MultipleConditionCode05ICD10', 'MultipleConditionCode06ICD10', 'MultipleConditionCode07ICD10', 'MultipleConditionCode08ICD10', 'MultipleConditionCode09ICD10', 'MultipleConditionCode10ICD10', 'CME_CircumstancesOtherText', 'CME_CrisisOtherDescription', 'LE_CircumstancesOtherText', 'LE_CrisisOtherDescription', 'SuicideAttemptHistory_c', 'SuicideThoughtHistory_c', 'HistorySelfHarm_c']
usemincol = ['IncidentID','DeathDate', 'DeathDate_myr', 'DeathDate_year', 'DeathState', 'DeathFIPS', 'IncidentCategory_c','PersonType', 'Sex', 'AgeYears_c'] 

uselesscols = ['IncidentID','IncidentYear', 'SiteID', 'IncidentNumber','NarrativeCME','NarrativeLE', 'IncidentCategory_c', 'HomicideSuicide_c', 
    'PersonID', 'VictimNumber', 'PersonType', 'NumberSuspects_c', 'Sex', 'AgeYears_c', 'Country', 'AbstractorDeathmanner_c', 'DeathDate', 'DeathDate_myr', 'DeathDate_year', 'DeathState', 'DeathFIPS']

if 'nvdrs_s_df' not in locals():
    nvdrs_df = load_nvdrs(file_key="nvdrs", data_folder="raw", usecols=usemincol )
    # Filter suicides
    nvdrs_s_df = filter_nvdrs_suicides(nvdrs_df)
    # starting from 01/01/2010
    nvdrs_s_df = nvdrs_s_df[nvdrs_s_df['DeathDate'] >= '2010-01-01']
    del nvdrs_df

nvdrs_s_daily_df = aggregate_nvdrs_daily(nvdrs_s_df, geo_level='county')


In [3]:
selected_cntys = {
#    'Arizona': ('Maricopa, AZ', 'Pima, AZ', 'Pinal, AZ', 'Yavapai, AZ', 'Coconino, AZ'),
    'Arizona': ('Pima, AZ', 'Pinal, AZ', 'Yavapai, AZ', 'Coconino, AZ'),  
    'Iowa': ('Polk, IA', 'Linn, IA', 'Scott, IA', 'Johnson, IA', 'Black Hawk, IA'), 
    'Kentucky': ('Jefferson, KY', 'Fayette, KY', 'Kenton, KY', 'Boone, KY', 'Warren, KY'), 
    'New Jersey': ('Bergen, NJ', 'Middlesex, NJ', 'Essex, NJ', 'Hudson, NJ', 'Monmouth, NJ'), 
    'New York': ('New York, NY', 'Kings, NY', 'Queens, NY', 'Bronx, NY', 'Richmond, NY'), 
    'North Carolina': ('Wake, NC', 'Mecklenburg, NC', 'Guilford, NC', 'Forsyth, NC', 'Cumberland, NC'), 
    'Utah': ('Salt Lake, UT', 'Utah, UT', 'Davis, UT', 'Weber, UT', 'Washington, UT')
}




#### 1. Case File 
- Location ID (FIPS)
- Number of Cases (e.g., 1)
- Date
- Optional: Age Group (for stratification)

#### 2. Population File (Required for Poisson model, skip if using Space-Time Permutation)
- Location ID (FIPS)
- Year
- Population Count
- Optional: Age Group (must match Case File)

#### 3. Coordinates File
- Location ID (FIPS)
- Latitude (or X coordinate)
- Longitude (or Y coordinate)

##### coordinates are "CENTERS OF POPULATION": 
`counties_pop_coord_2020.csv`: https://www2.census.gov/geo/docs/reference/cenpop2020/county/CenPop2020_Mean_CO.txt
###### The concept of the center of population as used by the U.S. Census Bureau is that of a balance point. The center of population is the point at which an imaginary, weightless, rigid, and flat (no elevation effects) surface representation of the 50 states (or 48 conterminous states for calculations made prior to 1960) and the District of Columbia would balance if weights of identical size were placed on it so that each weight represented the location of one person. https://www.census.gov/geographies/reference-files/time-series/geo/centers-population.html#accordion-07cb2673f6-item-043dc50b19
